# Source-release notes

Original-image vertex/frustration analysis and direction optimization.

Run this notebook with its containing directory as the kernel working directory. Experimental inputs and saved outputs are not included in this source release. See the [workflow guide](../docs/WORKFLOW.md), [data requirements](../docs/DATA.md), and [reproducibility notes](../docs/REPRODUCIBILITY.md) before execution.

Scientific settings are preserved. Packaging changes are recorded in [PREPARATION.md](../docs/PREPARATION.md). Original cell numbers in that record do not include this added introductory cell.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
import math
from sklearn.cluster import DBSCAN
from itertools import combinations

In [ ]:
mask_image_path = "{}/AB1.png".format(os.getcwd())
segmentation_visualization_path = "{}/bit_visualization_segments/reconstructed_segmentation_image.png".format(os.getcwd())

# Original image paths

original_segments_dir = "{}/mfm_segments".format(os.getcwd())
original_segments_with_arrows_dir = "{}/mfm_segments_with_arrows".format(os.getcwd())
reconstructed_original_path = "{}/reconstructed_mfm_image_with_directional_arrows.png".format(os.getcwd())
original_csv_path = "{}/mfm_segments_with_arrows/magnetic_moments.csv".format(os.getcwd())

# VAE image paths

synthetic_segments_dir = "{}/synthetic_mfm_segments".format(os.getcwd())
synthetic_segments_with_arrows_dir = "{}/synthetic_mfm_segments_with_arrows".format(os.getcwd())
reconstructed_synthetic_path = "{}/reconstructed_synthetic_mfm_image_with_directional_arrows.png".format(os.getcwd())
synthetic_csv_path = "{}/synthetic_mfm_segments_with_arrows/synthetic_magnetic_moments.csv".format(os.getcwd())

In [ ]:
# Load CSV files for arrow directions and strengths
original_df = pd.read_csv(original_csv_path)
synthetic_df = pd.read_csv(synthetic_csv_path)

# Load and display mask and reconstructed images for verification
mask_image = cv2.imread(mask_image_path, cv2.IMREAD_GRAYSCALE)
reconstructed_original = cv2.imread(reconstructed_original_path, cv2.IMREAD_COLOR)
reconstructed_synthetic = cv2.imread(reconstructed_synthetic_path, cv2.IMREAD_COLOR)

# Display images
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(mask_image, cmap="gray")
plt.title("Mask Image")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(reconstructed_original, cv2.COLOR_BGR2RGB))
plt.title("Reconstructed Original MFM Image")
plt.show()


# Identifying Vertices and Associating Segment Orientations

In [ ]:
# Define the directory path for the MFM segments
mfm_segments_with_arrows_dir = "mfm_segments_with_arrows"

# Example segment index
segment_index = 1

# Correct the path by avoiding any redundant "mfm_segment_" prefix and ".png" suffix
segment_path = os.path.join(mfm_segments_with_arrows_dir, f"mfm_segment_{segment_index}_with_arrow.png")

# Load and check if the segment loads successfully
segment_image = cv2.imread(segment_path, cv2.IMREAD_COLOR)
if segment_image is None:
    print(f"Failed to load segment {segment_index}. Path attempted: {segment_path}")
else:
    print(f"Successfully loaded segment {segment_index}.")


# Calculate each segment centers 

In [ ]:
# Directories for original segments and output
original_segments_dir = "{}/mfm_segments".format(os.getcwd())
output_dir = "{}/reconstructed_image".format(os.getcwd())
os.makedirs(output_dir, exist_ok=True)

# Path to the original "Bits Segmentation Visualization" image
segmentation_visualization_path = "bit_segmentation_visualization.png"

# Load the original segmentation visualization to get the original dimensions
segmentation_visualization = cv2.imread(segmentation_visualization_path)
if segmentation_visualization is None:
    print("Segmentation visualization image could not be loaded.")
else:
    # Create a blank canvas for the reconstructed image
    reassembled_image = np.zeros_like(segmentation_visualization)

    # Convert to grayscale and find connected components in the segmentation visualization
    segmentation_gray = cv2.cvtColor(segmentation_visualization, cv2.COLOR_BGR2GRAY)
    _, binary_mask = cv2.threshold(segmentation_gray, 1, 255, cv2.THRESH_BINARY)
    num_labels, labels_im = cv2.connectedComponents(binary_mask)

    # Placeholder for storing segment positions
    segment_positions = []
    min_segment_area = 50  # Minimum area to be considered a valid segment (adjust as needed)

    # Loop over each segment label in the mask
    for label in range(1, num_labels):
        # Path for the original segment
        original_segment_path = os.path.join(original_segments_dir, f"mfm_segment_{label}.png")

        # Load the original segment
        original_segment = cv2.imread(original_segment_path, cv2.IMREAD_COLOR)

        if original_segment is None:
            print(f"Original Segment {label} could not be loaded. Skipping...")
            continue

        # Find the bounding box of the segment in the mask
        mask = (labels_im == label).astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            print(f"No contours found for segment {label}. Skipping...")
            continue

        # Get the bounding box of the current segment area in the mask
        x, y, w, h = cv2.boundingRect(contours[0])

        # Calculate area and filter out small regions
        area = w * h
        if area < min_segment_area:
            print(f"Skipping segment {label} due to small area: {area}")
            continue

        # Overlay the original segment onto the reassembled image (no resizing needed if they're exactly the same size)
        reassembled_image[y:y + h, x:x + w] = cv2.resize(original_segment, (w, h))

        # Calculate the center of the segment and add to the list
        x_center = x + w // 2
        y_center = y + h // 2
        segment_positions.append((label, x_center, y_center))

        # Print for debugging
        print(f"Segment {label}: Center at ({x_center}, {y_center}), Area: {area}")

    # Step 2: Display the reassembled image with segment centers marked
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(reassembled_image, cv2.COLOR_BGR2RGB))
    plt.scatter([pos[1] for pos in segment_positions], [pos[2] for pos in segment_positions], color='red', s=50, label='Segment Centers')
    plt.legend()
    plt.title("Reconstructed Original MFM Image with Segment Centers")
    plt.axis('off')
    plt.show()

    # Step 3: Save segment positions to CSV for future reference
    positions_df = pd.DataFrame(segment_positions, columns=['Segment', 'x', 'y'])
    csv_output_path = os.path.join(output_dir, "original_segment_positions.csv")
    positions_df.to_csv(csv_output_path, index=False)
    print(f"Segment positions saved to {csv_output_path}")


# Determine the vertices of the sample

In [ ]:
# Load the CSV with segment positions
positions_csv_path = "{}/reconstructed_image/original_segment_positions.csv".format(os.getcwd())
segment_positions_df = pd.read_csv(positions_csv_path)

# Parameters
radius = 30  # Maximum distance to consider segments as part of a vertex

# Placeholder for storing vertices (centroid of three segments forming a triangle)
vertex_positions = []
vertex_segments = []

# Iterate through all possible triplets of segments
for (i, row1), (j, row2), (k, row3) in combinations(segment_positions_df.iterrows(), 3):
    segment1 = row1['Segment']
    segment2 = row2['Segment']
    segment3 = row3['Segment']
    
    # Get segment positions
    pos1 = (row1['x'], row1['y'])
    pos2 = (row2['x'], row2['y'])
    pos3 = (row3['x'], row3['y'])
    
    # Calculate distances between each pair of segments
    dist12 = np.linalg.norm(np.array(pos1) - np.array(pos2))
    dist13 = np.linalg.norm(np.array(pos1) - np.array(pos3))
    dist23 = np.linalg.norm(np.array(pos2) - np.array(pos3))
    
    # Check if all distances are within the radius threshold
    if dist12 <= radius and dist13 <= radius and dist23 <= radius:
        # Calculate the centroid (average position) of the triangle formed by these three segments
        x_center = int((pos1[0] + pos2[0] + pos3[0]) / 3)
        y_center = int((pos1[1] + pos2[1] + pos3[1]) / 3)
        
        # Store the centroid and associated segments
        vertex_positions.append((x_center, y_center))
        vertex_segments.append([segment1, segment2, segment3])

# Plot the vertices on the reconstructed image
reconstructed_image = cv2.imread(reconstructed_original_path)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(reconstructed_image, cv2.COLOR_BGR2RGB))

# Mark each vertex and plot associated segment centers
for i, (vertex, segments) in enumerate(zip(vertex_positions, vertex_segments)):
    # Plot the centroid (vertex)
    plt.scatter(*vertex, color='blue', s=100, marker='o', label=f'Vertex {i+1}' if i == 0 else "")
    
    # Plot centers of the segments forming the vertex
    for segment_id in segments:
        segment_info = segment_positions_df[segment_positions_df['Segment'] == segment_id]
        plt.scatter(segment_info['x'].values[0], segment_info['y'].values[0], color='red', s=50)

    # Draw lines between the segment centers and the centroid for visualization
    for segment_id in segments:
        segment_info = segment_positions_df[segment_positions_df['Segment'] == segment_id]
        plt.plot([vertex[0], segment_info['x'].values[0]], [vertex[1], segment_info['y'].values[0]], color='blue', linestyle='--')

plt.legend(['Vertices', 'Segment Centers'])
plt.title("Detected Vertices as Centroids of Segment Triangles")
plt.axis('off')
plt.show()

# Save vertices with associated segments to a CSV
vertices_df = pd.DataFrame({
    'Vertex': range(1, len(vertex_positions) + 1),
    'x': [pos[0] for pos in vertex_positions],
    'y': [pos[1] for pos in vertex_positions],
    'Associated_Segments': [','.join(map(str, segs)) for segs in vertex_segments]
})
vertices_output_csv_path = "{}/reconstructed_image/vertices_with_3_segments.csv".format(os.getcwd())
vertices_df.to_csv(vertices_output_csv_path, index=False)
print(f"Vertices saved to {vertices_output_csv_path}")


# Calculate the Magnetic Frustration

In [ ]:
# Load the vertices data and magnetic moments data
vertices_df = pd.read_csv("{}/reconstructed_image/vertices_with_3_segments.csv".format(os.getcwd()))
magnetic_moments_df = pd.read_csv("{}/mfm_segments_with_arrows/magnetic_moments.csv".format(os.getcwd()))


In [ ]:
# File paths
vertices_file_path = "{}/reconstructed_image/vertices_with_3_segments.csv".format(os.getcwd())
magnetic_moments_file_path = "{}/mfm_segments_with_arrows/magnetic_moments.csv".format(os.getcwd())
original_segment_positions_path = "{}/reconstructed_image/original_segment_positions.csv".format(os.getcwd())

# Load data
vertices_df = pd.read_csv(vertices_file_path)
magnetic_moments_df = pd.read_csv(magnetic_moments_file_path)
original_segment_positions = pd.read_csv(original_segment_positions_path)

# Normalize the 'Segment' column in magnetic_moments_df
magnetic_moments_df['Segment'] = magnetic_moments_df['Segment'].str.extract(r'mfm_segment_(\d+)\.png').astype(int)

# Create a mapping of segment to direction
segment_directions = magnetic_moments_df[['Segment', 'Direction']].set_index('Segment').to_dict()['Direction']

# Create a mapping of segment to centroid position
segment_positions = original_segment_positions.set_index('Segment')[['x', 'y']].to_dict(orient='index')

# Create a mapping of segment to its corresponding vertex
segment_to_vertex_mapping = {}
for _, vertex_row in vertices_df.iterrows():
    vertex_id = vertex_row['Vertex']
    segments = [int(seg) for seg in vertex_row['Associated_Segments'].split(',')]
    for segment in segments:
        segment_to_vertex_mapping.setdefault(segment, []).append(vertex_id)

# Function to find the orientation of a segment
def get_segment_orientation(segment_path):
    segment = cv2.imread(segment_path, cv2.IMREAD_GRAYSCALE)
    if segment is None:
        return None, None  # Segment missing

    # Threshold and find contours
    _, binary = cv2.threshold(segment, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Filter contours by area
    min_contour_area = 50
    contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_contour_area]

    if not contours:
        return None, None

    # Fit an ellipse to find orientation if there are enough points
    if len(contours[0]) >= 5:
        ellipse = cv2.fitEllipse(contours[0])
        angle = ellipse[2]  # Orientation angle in degrees
        return angle, cv2.boundingRect(contours[0])  # Return angle and bounding box

    return None, None


# Function to classify frustration status
def classify_detailed_frustration(vertex_id, segments):
    horizontal_segment = None

    # Identify the horizontal segment (angle close to 0° or 90°)
    for segment in segments:
        segment_path = f"{os.getcwd()}/mfm_segments_with_arrows/mfm_segment_{segment}_with_arrow.png"
        angle, _ = get_segment_orientation(segment_path)
        if angle is not None and (abs(angle) < 15 or abs(angle - 90) < 15):  # Correct angle detection
            horizontal_segment = segment
            break

    if horizontal_segment is None:
        print(f"Warning: No horizontal segment found for Vertex {vertex_id}")
        return "Missing Data", "gray", "o"

    # Get the direction of the horizontal segment
    x = segment_directions[horizontal_segment]  # Direction of the horizontal segment
    angled_segments = [seg for seg in segments if seg != horizontal_segment]
    y = segment_directions[angled_segments[0]]
    z = segment_directions[angled_segments[1]]

    # Get the vertex and segment positions
    vertex_position = vertices_df.loc[vertices_df['Vertex'] == vertex_id, ['x', 'y']].iloc[0]
    segment_position = segment_positions[horizontal_segment]

    # Check if frustrated
    if y == z and y != x:  # Frustrated condition
        # Find associated vertices of the horizontal segment
        associated_vertices = segment_to_vertex_mapping.get(horizontal_segment, [])

        # Debugging: Check vertex associations
        print(f"Debug: Vertex {vertex_id} -> x={x}, y={y}, z={z}, Associated_Vertices={associated_vertices}")

        # If two vertices share the horizontal segment
        if len(associated_vertices) == 2:
            vertex_positions = {
                v_id: (vertices_df.loc[vertices_df['Vertex'] == v_id, 'x'].values[0],
                       vertices_df.loc[vertices_df['Vertex'] == v_id, 'y'].values[0])
                for v_id in associated_vertices
            }

            # Determine the "Dark" and "Bright" vertices based on x-coordinate
            for v_id, pos in vertex_positions.items():
                if pos[0] > segment_position['x']:  # Vertex is to the right of the segment
                    if v_id == vertex_id:
                        return "Dark Frustrated", "red", "s"
                else:  # Vertex is to the left of the segment
                    if v_id == vertex_id:
                        return "Bright Frustrated", "magenta", "s"

        # If only one vertex is associated with the horizontal segment
        elif len(associated_vertices) == 1:
            # Compare vertex position to the horizontal segment position
            if vertex_position['x'] < segment_position['x'] and x == 0:
                return "Dark Frustrated", "red", "s"
            elif vertex_position['x'] > segment_position['x'] and x == 1:
                return "Bright Frustrated", "magenta", "s"

    # Subclassify Unfrustrated vertices
    elif y == z == x:  # All directions match
        # Use the position relative to the horizontal segment for classification
        if vertex_position['x'] < segment_position['x']:
            if x == 1:  # Horizontal segment direction points right
                return "Dark Unfrustrated", "yellow", "*"
            elif x == 0:  # Horizontal segment direction points left
                return "Bright Unfrustrated", "green", "*"
        elif vertex_position['x'] > segment_position['x']:
            if x == 1:  # Horizontal segment direction points right
                return "Bright Unfrustrated", "green", "*"
            elif x == 0:  # Horizontal segment direction points left
                return "Dark Unfrustrated", "yellow", "*"

    # Additional logic for unfrustrated cases where y != z
    if vertex_position['x'] < segment_position['x'] and x == 1 and y != z:
        return "Bright Unfrustrated", "green", "*"
    elif vertex_position['x'] < segment_position['x'] and x == 0 and y != z:
        return "Dark Unfrustrated", "yellow", "*"
    elif vertex_position['x'] > segment_position['x'] and x == 1 and y != z:
        return "Dark Unfrustrated", "yellow", "*"
    elif vertex_position['x'] > segment_position['x'] and x == 0 and y != z:
        return "Bright Unfrustrated", "green", "*"

    return "Unclassified", "gray", "o"


# Calculate frustration classification for each vertex
vertex_classification_status = []

for _, row in vertices_df.iterrows():
    vertex_id = row['Vertex']
    segments = [int(seg) for seg in row['Associated_Segments'].split(',')]

    # Classify frustration
    classification_status, color, marker = classify_detailed_frustration(vertex_id, segments)

    # Append results
    vertex_classification_status.append({
        "Vertex": vertex_id,
        "Position": (row['x'], row['y']),
        "Classification_Status": classification_status,
        "Color": color,
        "Marker": marker
    })

# Convert to DataFrame for analysis
classification_df = pd.DataFrame(vertex_classification_status)

# # Save to CSV for future reference
# classification_output_csv_path = "{}/vertex_detailed_classification_analysis.csv".format(os.getcwd())
# classification_df.to_csv(classification_output_csv_path, index=False)
# print(f"Classification analysis saved to {classification_output_csv_path}")

# Print summary of classifications
for classification in classification_df["Classification_Status"].unique():
    count = len(classification_df[classification_df["Classification_Status"] == classification])
    print(f"{classification}: {count} vertices")

# # Visualization of detailed classification
# plt.figure(figsize=(10, 10))

# # Assuming `reconstructed_synthetic` is loaded as an image
# plt.imshow(cv2.cvtColor(reconstructed_original, cv2.COLOR_BGR2RGB))

# # Plot vertices color-coded by detailed classification
# for _, row in classification_df.iterrows():
#     plt.scatter(
#         row['Position'][0],
#         row['Position'][1],
#         color=row['Color'],
#         marker=row['Marker'],
#         s=100
#     )

# plt.title("Detailed Frustration Classification of Vertices")
# plt.axis('off')
# plt.show()

# Visualization of detailed classification
plt.figure(figsize=(10, 10))  # Create a new figure

# Assuming `reconstructed_synthetic` is loaded as an image
plt.imshow(cv2.cvtColor(reconstructed_original, cv2.COLOR_BGR2RGB))

# Plot vertices color-coded by detailed classification
for _, row in classification_df.iterrows():
    plt.scatter(
        row['Position'][0],
        row['Position'][1],
        color=row['Color'],
        marker=row['Marker'],
        s=100
    )

plt.title("Detailed Frustration Classification of Vertices")
plt.axis('off')

# Save the plot as an image in the current working directory
output_image_path = os.path.join(os.getcwd(), "frustration_classification_original_mfm.png")
plt.savefig(output_image_path, bbox_inches='tight', dpi=300)  # Save with high resolution
print(f"Image saved to {output_image_path}")

plt.show()





# Optimizing magnetic moments to reduce frustrations

In [ ]:
# Save the optimized directions
optimized_output_csv_path = "{}/optimized_magnetic_moments.csv".format(os.getcwd())
magnetic_moments_df.to_csv(optimized_output_csv_path, index=False)
print(f"Optimized magnetic moments saved to {optimized_output_csv_path}")

# Summary of frustration status
vertex_frustration_status = []
for _, vertex_row in vertices_df.iterrows():
    vertex_id = vertex_row['Vertex']
    vertex_type, frustration_status = evaluate_vertex(vertex_row, magnetic_moments_df)
    vertex_frustration_status.append({
        "Vertex": vertex_id,
        "Type": vertex_type,
        "Frustration_Status": frustration_status
    })

frustration_summary_df = pd.DataFrame(vertex_frustration_status)

# Print summary
frustrated_vertices = frustration_summary_df[frustration_summary_df["Frustration_Status"] == "Frustrated"]
unfrustrated_vertices = frustration_summary_df[frustration_summary_df["Frustration_Status"] == "Unfrustrated"]

print(f"Total Vertices after optimization: {len(vertices_df)}")
print(f"Frustrated Vertices after optimization: {len(frustrated_vertices)}")
print(f"Unfrustrated Vertices after optimization: {len(unfrustrated_vertices)}")

# Save the frustration summary
frustration_summary_csv_path = "{}/optimized_vertex_frustration_summary.csv".format(os.getcwd())
frustration_summary_df.to_csv(frustration_summary_csv_path, index=False)
print(f"Frustration summary saved to {frustration_summary_csv_path}")


# Optimizaion based on New frustration Calculation

In [ ]:

# Load the vertices and magnetic moments data
vertices_df = pd.read_csv("{}/reconstructed_image/vertices_with_3_segments.csv".format(os.getcwd()))
magnetic_moments_df = pd.read_csv("{}/mfm_segments_with_arrows/magnetic_moments.csv".format(os.getcwd()))

# Normalize the 'Segment' column in magnetic_moments_df to match integer format in vertices_df
magnetic_moments_df['Segment'] = magnetic_moments_df['Segment'].astype(str).str.extract(r'mfm_segment_(\d+)\.png').astype(float).astype('Int64')

# Add a 'Toggle' column initialized to False to track toggled segments
magnetic_moments_df['Toggle'] = False

# Function to determine if a segment is aligned with the x-axis based on width and height
def is_aligned_with_x_axis(segment_path):
    segment = cv2.imread(segment_path, cv2.IMREAD_GRAYSCALE)
    if segment is None:
        return False  # Handle cases where segment image is missing
    
    # Threshold and find contours
    _, binary = cv2.threshold(segment, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return False
    
    # Get bounding box of the contour
    x, y, w, h = cv2.boundingRect(contours[0])
    return w > h  # If width is greater than height, we assume alignment with x-axis

# Define frustration calculation function with specific condition
def calculate_frustration_status(inward_count, outward_count, segments):
    if inward_count == 2 and outward_count == 1:
        for segment_id in segments:
            segment_data = magnetic_moments_df[magnetic_moments_df['Segment'] == segment_id]
            direction = segment_data['Direction'].values[0]
            segment_path = f"{os.getcwd()}/mfm_segments_with_arrows/mfm_segment_{segment_id}_with_arrow.png"
            if direction == 1 and is_aligned_with_x_axis(segment_path):
                return "Unfrustrated"
        return "Frustrated"
    elif inward_count == 3 and outward_count == 0:
        return "Unfrustrated"
    elif inward_count == 0 and outward_count == 3:
        return "Frustrated"
    return "Frustrated" if abs(inward_count - outward_count) <= 1 else "Unfrustrated"

# Evaluate a vertex to determine frustration status based on modified logic
def evaluate_vertex(vertex_row, magnetic_moments_df):
    segments = [int(seg) for seg in vertex_row['Associated_Segments'].split(',')]
    inward_count = 0
    outward_count = 0
    
    for segment_id in segments:
        segment_data = magnetic_moments_df[magnetic_moments_df['Segment'] == segment_id]
        
        if segment_data.empty:
            print(f"Warning: Segment ID {segment_id} not found in magnetic_moments_df.")
            continue
        
        direction = segment_data['Direction'].values[0]
        
        if direction == 1:
            inward_count += 1
        elif direction == 0:
            outward_count += 1
    
    # Calculate frustration status using the provided logic
    frustration_status = calculate_frustration_status(inward_count, outward_count, segments)
    return "Frustrated" if frustration_status == "Frustrated" else "Unfrustrated"

# Optimization loop to reduce frustration
frustration_reduced = True
while frustration_reduced:
    frustration_reduced = False

    for _, vertex_row in vertices_df.iterrows():
        vertex_id = vertex_row['Vertex']
        segments = [int(seg) for seg in vertex_row['Associated_Segments'].split(',')]
        
        original_inward_count = 0
        original_outward_count = 0
        for segment_id in segments:
            direction = magnetic_moments_df.loc[magnetic_moments_df['Segment'] == segment_id, 'Direction'].values[0]
            if direction == 1:
                original_inward_count += 1
            elif direction == 0:
                original_outward_count += 1

        original_frustration = calculate_frustration_status(original_inward_count, original_outward_count, segments)

        vertex_optimized = False
        for segment_id in segments:
            current_direction = magnetic_moments_df.loc[magnetic_moments_df['Segment'] == segment_id, 'Direction'].values[0]
            magnetic_moments_df.loc[magnetic_moments_df['Segment'] == segment_id, 'Direction'] = 1 - current_direction

            inward_count, outward_count = 0, 0
            for seg in segments:
                direction = magnetic_moments_df.loc[magnetic_moments_df['Segment'] == seg, 'Direction'].values[0]
                if direction == 1:
                    inward_count += 1
                elif direction == 0:
                    outward_count += 1

            new_frustration = calculate_frustration_status(inward_count, outward_count, segments)

            if new_frustration == "Unfrustrated" and original_frustration == "Frustrated":
                print(f"Vertex {vertex_id}: Frustration reduced by toggling segment {segment_id}.")
                magnetic_moments_df.loc[magnetic_moments_df['Segment'] == segment_id, 'Toggle'] = True
                frustration_reduced = True
                original_frustration = new_frustration
                vertex_optimized = True
            else:
                magnetic_moments_df.loc[magnetic_moments_df['Segment'] == segment_id, 'Direction'] = current_direction

        if vertex_optimized:
            frustration_reduced = True

# Save the optimized magnetic moments data with the Toggle column
optimized_output_csv_path = "{}/optimized_magnetic_moments.csv".format(os.getcwd())
magnetic_moments_df.to_csv(optimized_output_csv_path, index=False)
print(f"Optimized magnetic moments saved to {optimized_output_csv_path}")

# Summary of frustration status
vertex_frustration_status = []
for _, vertex_row in vertices_df.iterrows():
    vertex_id = vertex_row['Vertex']
    frustration_status = evaluate_vertex(vertex_row, magnetic_moments_df)
    vertex_frustration_status.append({
        "Vertex": vertex_id,
        "Frustration_Status": frustration_status
    })

frustration_summary_df = pd.DataFrame(vertex_frustration_status)

# Print summary
frustrated_vertices = frustration_summary_df[frustration_summary_df["Frustration_Status"] == "Frustrated"]
unfrustrated_vertices = frustration_summary_df[frustration_summary_df["Frustration_Status"] == "Unfrustrated"]

print(f"Total Vertices after optimization: {len(vertices_df)}")
print(f"Frustrated Vertices after optimization: {len(frustrated_vertices)}")
print(f"Unfrustrated Vertices after optimization: {len(unfrustrated_vertices)}")

# Save the frustration summary
frustration_summary_csv_path = "{}/optimized_vertex_frustration_summary.csv".format(os.getcwd())
frustration_summary_df.to_csv(frustration_summary_csv_path, index=False)
print(f"Frustration summary saved to {frustration_summary_csv_path}")


# After Optimizing the vertex

In [ ]:
# Load pre-optimization vertex frustration data
pre_optimization_df = pd.read_csv("{}/vertex_frustration_analysis.csv".format(os.getcwd()))

# Load the optimized magnetic moments and segment positions
optimized_magnetic_moments_df = pd.read_csv("{}/optimized_magnetic_moments.csv".format(os.getcwd()))
segment_positions_df = pd.read_csv("{}/reconstructed_image/original_segment_positions.csv".format(os.getcwd()))

# Extract the toggled segments and merge with their x, y positions
toggled_segments = optimized_magnetic_moments_df[optimized_magnetic_moments_df['Toggle'] == True]
toggled_segments = toggled_segments.merge(segment_positions_df, on='Segment', how='left')

# Load the background image for visualization
#reconstructed_image_path = "reconstructed_image.png"  # Adjust the path if necessary
reconstructed_image = cv2.imread(reconstructed_original_path)
if reconstructed_image is None:
    print("Error: Background image could not be loaded.")
else:
    reconstructed_image = cv2.cvtColor(reconstructed_image, cv2.COLOR_BGR2RGB)

# Plot the background image
plt.figure(figsize=(12, 12))
plt.imshow(reconstructed_image)
plt.axis("off")

# Plot each vertex with the original (pre-optimization) frustration status
for _, row in pre_optimization_df.iterrows():
    vertex_position = eval(row['Position'])  # Convert string "(x, y)" to tuple
    frustration_status = row['Frustration_Status']
    
    # Set color based on frustration status before optimization
    color = 'red' if frustration_status == "Frustrated" else 'green'
    plt.scatter(vertex_position[0], vertex_position[1], color=color, s=100, label=frustration_status if row['Vertex'] == 1 else "")

# Plot the centers of the toggled segments
for _, segment_row in toggled_segments.iterrows():
    segment_x = segment_row['x']
    segment_y = segment_row['y']
    
    # Highlight toggled segments with a blue marker
    plt.scatter(segment_x, segment_y, color='blue', s=150, edgecolor='yellow', marker='s', label="Toggled Segment" if segment_row.name == 0 else "")

# Add legend
plt.legend(loc="upper right", markerscale=1.5)
plt.title("Frustration Analysis with Toggled Segments Highlighted\n(Red=Original Frustrated, Green=Original Unfrustrated, Blue=Toggled Segment)")
plt.show()
